# Practical 14 --- Teaching the Model the New Suburb

**Retrain on the right window, judge old against new on the same holdout, promote with a way back**

SCSE3040 Machine Learning Operations &middot; Bennett University &middot; Session 2026-27

| | |
|---|---|
| **Lectures this follows** | L27-L28 |
| **Course Outcome** | CO5 |
| **Lab duration** | 120 minutes |
| **Memory needed** | about 600 MB (fine on a 4 GB or 8 GB machine) |
| **Extra software** | nothing beyond the course venv |
| **Marks** | 10 |

---

## Aim

1. Choose which rows a retrained model should learn from, and say why.
2. Judge the old and the new model on the same held-out orders, never on training rows.
3. Write the promotion gate as code: beat the bar, and beat the old model.
4. Register the new version, move the alias, and keep the old version for rollback.

---

## What you need to know first


P13 ended with an alarm. The delivery company opened a new suburb, trips got
longer, and the model's error tripled. The audit found the reason: past 12 km a
rider is on the highway, every extra kilometre costs more, and the training data
had never contained such a trip. The honest fix was named but not done:
**retrain**.

Retraining sounds like "run the training cell again". It is not, and this
practical is about the four decisions that separate a real retrain from that.

**Which rows?** Not all history --- most of it describes a city that no longer
exists. Not last week alone --- too few rows. The **recent window**, after the
world settled into its new shape. The rows from the new suburb are now the most
valuable rows you own.

**Judged how?** Both models, old and new, on the **same held-out orders** that
neither has trained on. A model judged on its own training rows always wins,
and always lies.

**Promoted when?** Two conditions, both required. The new model must **beat the
bar** --- a fixed quality the product promised --- and it must **beat the old
model**. Passing one is not enough. This is the **gate**, and today you write it
as a function so that it cannot be argued with at eleven at night.

**Replaced how?** The new model becomes **version 2** in the registry. The
alias `@champion` moves from 1 to 2. Version 1 is **kept**, because the way back
is the whole point of versions. The service loads `@champion` by name, so
nothing in the serving code changes.

The pipeline *proposes*; a named person *promotes*. Automation earns trust
slowly, and the alias stays a human decision in this course.

---

## Before you start

- Practical P13 is finished. You watched this model get worse and found who it served worst: long trips.
- Practical P06 is finished. You remember MLflow's registry: a model name, numbered versions, and loading by name.
- You remember MAE from P02: the average miss, in minutes.

### How to run a cell

Click on a grey code cell, then press **Shift + Enter**. The cell runs and
the cursor moves to the next one. A number appears in the `[ ]` on the left
when the cell has finished.

**Run the cells in order, from the top.** A later cell almost always uses
something an earlier cell created. If you jump ahead you will see a
`NameError`, which just means "you have not made that thing yet".

If everything goes wrong, use the menu: **Kernel -> Restart Kernel and Clear
All Outputs**, then start again from the first cell. Nothing is damaged by
doing this.

In [ ]:
# Step 0 --- check the workbench before we start.
# This cell only looks; it changes nothing. Run it and read the last line.

import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['numpy', 'pandas', 'sklearn', 'mlflow', 'matplotlib']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['numpy', 'pandas', 'sklearn', 'mlflow', 'matplotlib']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

---

## Step 0b --- the dataset

Every practical in this course uses the same 600 food deliveries.
The next cell makes sure the file is there.

In [ ]:
# The delivery-time dataset every practical in this course uses.
# If the file is missing we build it again from the same seed, so every
# student in the room gets byte-for-byte the same 600 rows.

import csv
from pathlib import Path

import numpy as np

SEED = 42
N_ROWS = 600
DATA = Path("..") / "data" / "delivery_times.csv"


def make_delivery_csv(path=DATA):
    """Write the 600-row delivery dataset. Same formula as the lectures."""
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level",
                    "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]),
                        int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
    print("dataset rebuilt ->", DATA)
else:
    print("dataset found   ->", DATA)

---

# Walkthrough

Read each step, then run its cell.

### Step 1 --- Train the model that raised the alarm

The launch model from P13, trained on the original 600 orders. Its
test error at launch is the quality the product promised its
customers.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

WORK = Path("work")
WORK.mkdir(exist_ok=True)
FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]

orders = pd.read_csv(DATA)
X_train, X_test, y_train, y_test = train_test_split(
    orders[FEATURES], orders["delivery_min"],
    test_size=0.2, random_state=42,
)
model = LinearRegression().fit(X_train, y_train)

launch_mae = mean_absolute_error(y_test, model.predict(X_test))
print(f"MAE at launch: {launch_mae:.2f} minutes")

### Step 2 --- Twelve weeks of traffic, and the alarm

The same traffic generator as P13, run for twelve weeks instead of
eight. Weeks 1 to 4 look like training. From week 5 the suburb
opens and trips stretch. **From week 9 the world settles**: the
suburb is simply part of the city now, and the shape of the data
stops changing.

The alarm rule from P13 --- error above 1.5 times the launch
figure, two weeks running --- is applied at the end.

In [ ]:
SEED = 7
rng = np.random.default_rng(SEED)
ORDERS_PER_WEEK = 150


def week_of_orders(week):
    """One week of live traffic. Trips stretch from week 5, settle at 9."""
    stretch = 1.0 if week <= 4 else min(1.0 + 0.22 * (week - 4), 1.88)

    distance_km = np.round(
        rng.uniform(0.5, 12.0, ORDERS_PER_WEEK) * stretch, 2)
    prep_time_min = np.round(
        rng.uniform(5, 30, ORDERS_PER_WEEK), 0).astype(int)
    traffic_level = rng.integers(1, 4, ORDERS_PER_WEEK)
    rain = rng.binomial(1, 0.25, ORDERS_PER_WEEK)

    # Beyond 12 km the rider is on the highway and every extra
    # kilometre costs more. The launch model never saw this rule.
    long_trip_penalty = 1.8 * np.maximum(distance_km - 12.0, 0.0)

    true_minutes = np.round(
        6.0 + 3.1 * distance_km + 0.65 * prep_time_min
        + 4.2 * traffic_level + 5.5 * rain
        + long_trip_penalty
        + rng.normal(0, 2.5, ORDERS_PER_WEEK), 1)

    return pd.DataFrame({
        "week": week,
        "distance_km": distance_km,
        "prep_time_min": prep_time_min,
        "traffic_level": traffic_level,
        "rain": rain,
        "actual_min": true_minutes,
    })


live = pd.concat([week_of_orders(w) for w in range(1, 13)],
                 ignore_index=True)
live["predicted_min"] = model.predict(live[FEATURES])
live["error"] = (live["actual_min"] - live["predicted_min"]).abs()

weekly = (live.groupby("week")
          .agg(orders=("error", "size"),
               mae=("error", "mean"),
               mean_distance=("distance_km", "mean"))
          .round(2))
print(weekly.to_string())

threshold = launch_mae * 1.5
streak, first_alert = 0, None
for week, row in weekly.iterrows():
    streak = streak + 1 if row["mae"] > threshold else 0
    if streak >= 2 and first_alert is None:
        first_alert = int(week)
print()
print(f"alert threshold {threshold:.2f}; the alarm fired in week "
      f"{first_alert}")

Weeks 9 to 12 all sit near 6 minutes of error, three times the
launch figure, and their average distance has stopped climbing.
The world has moved and then stopped moving. That is the moment a
retrain makes sense: retraining *while* the data is still changing
just chases it.

### Step 3 --- Draw the timeline, and mark the window

The same story as a picture, with the retrain window shaded. This
is the chart you would put in front of the person who has to
approve the retrain.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.axvspan(8.5, 12.5, color="#2C5F79", alpha=0.08,
           label="retrain window: weeks 9-12")
ax.plot(weekly.index, weekly["mae"], marker="o", linewidth=2,
        color="#9E3B33", label="error of the launch model")
ax.axhline(launch_mae, linestyle="--", color="#6B7785",
           label=f"launch MAE {launch_mae:.2f}")
ax.axhline(threshold, linestyle=":", color="#6B7785",
           label=f"alert threshold {threshold:.2f}")
ax.annotate("alarm", (first_alert, weekly.loc[first_alert, "mae"]),
            xytext=(-38, 28), textcoords="offset points",
            color="#9E3B33",
            arrowprops=dict(arrowstyle="->", color="#9E3B33"))
ax.set_xticks(range(1, 13))
ax.set_xlabel("week")
ax.set_ylabel("MAE (minutes)")
ax.set_title("The error climbed, then settled high. "
             "The shaded weeks are the ones worth learning from.")
ax.legend(fontsize=8, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### Step 4 --- Has the world stopped moving? Check before you retrain

The shaded window is a claim: "weeks 9 to 12 describe the world as
it is now". Check the claim before you build on it.

The question is: **which past weeks look like this week?** We take
week 12 as "now" and measure every earlier week against it with
**PSI** (Population Stability Index), the drift score from P13.
Below 0.25 means "no real shift": the same shape as now.

The PSI values span from about 0.05 to about 6, so the chart uses
a **log scale**: each grid line is ten times the one below it.

The right-hand chart asks a second question: does the window
actually **contain** the trips the launch model never saw?

In [ ]:
def psi(reference, current, buckets=10):
    """Population Stability Index between two samples (from P13)."""
    edges = np.percentile(reference, np.linspace(0, 100, buckets + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    ref_share = np.histogram(reference, bins=edges)[0] / len(reference)
    cur_share = np.histogram(current, bins=edges)[0] / len(current)
    ref_share = np.clip(ref_share, 1e-6, None)
    cur_share = np.clip(cur_share, 1e-6, None)
    return float(np.sum((cur_share - ref_share)
                        * np.log(cur_share / ref_share)))


now = live.loc[live["week"] == 12, "distance_km"]
like_now = pd.Series({w: psi(now, live.loc[live["week"] == w,
                                         "distance_km"])
                      for w in range(1, 12)})

window = live[live["week"] >= 9]
beyond_train = (X_train["distance_km"] > 12).mean()
beyond_window = (window["distance_km"] > 12).mean()

print("PSI of each week's distances against week 12:")
print(like_now.round(3).to_string())
print()
print(f"trips over 12 km in the launch training data: {beyond_train:.0%}")
print(f"trips over 12 km in the weeks 9-12 window:    {beyond_window:.0%}")

fig, (left, right) = plt.subplots(
    1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1.3, 1]})

left.bar(like_now.index, like_now.values,
         color=["#2C5F79" if v < 0.25 else "#B9BFC7"
                for v in like_now])
left.axhline(0.25, color="#6B7785", linestyle=":", linewidth=1)
left.text(0.6, 0.3, "0.25: a real shift (P13)", fontsize=8,
          color="#6B7785")
left.set_yscale("log")
left.set_xticks(range(1, 12))
left.set_xlabel("week")
left.set_ylabel("PSI against week 12")
left.set_title("Which weeks look like now? (blue = yes)",
               fontsize=10)
left.spines[["top", "right"]].set_visible(False)

edges = np.linspace(0, 24, 25)
right.hist(X_train["distance_km"], bins=edges, density=True,
           color="#B9BFC7", label="launch training data")
right.hist(window["distance_km"], bins=edges, density=True,
           color="#2C5F79", alpha=0.6, label="weeks 9-12 window")
right.axvline(12, color="#6B7785", linestyle=":", linewidth=1)
right.set_xlabel("distance_km")
right.set_ylabel("share of orders")
right.set_title("What each set of rows can teach", fontsize=10)
right.legend(fontsize=8)
right.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

**Left chart.** Weeks 1 to 7 are grey and far above the line: they
describe a city that no longer exists. Weeks 8 to 11 are blue and
far below it: every one has the same shape as now. The small
values they do have, around 0.05 to 0.14, are the same ordinary
noise as P13's week-3 false alarm. The world moved, then stopped.
That is the evidence for the window.

Why not include week 8, which is also blue? You could. We leave it
out to keep a margin from the weeks when things were still moving;
either choice is defensible, and the point is that you can now
*argue* it from a chart.

**Right chart.** The grey shape stops dead at 12 km: the launch
model had nothing to learn from out there. The blue shape has a
long tail past the line. Those rows are the most valuable rows you
own, because they are the only ones that can teach the highway
rule.

### Step 5 --- Choose the window, and hold out before you touch anything

The recent window is weeks 9 to 12: 600 orders from the settled
new world. Before any model sees them, **seal 20% away** as the
holdout. Every candidate today --- old, new, and the ones you build
in the tasks --- is judged on exactly these sealed orders.

This is P02's train/test split again, with one extra rule: the
*old* model is judged on the same holdout too. Same exam paper for
everyone.

In [ ]:
recent = live[live["week"] >= 9].copy()

train, holdout = train_test_split(recent, test_size=0.2,
                                  random_state=42)

print(f"recent window : weeks 9-12, {len(recent)} orders")
print(f"train on      : {len(train)} orders")
print(f"hold out      : {len(holdout)} orders  <- sealed until judging")

### Step 6 --- Candidate A: the same code, the new window

L28's recipe: change nothing about the training code, only the
rows it learns from. Then judge the old model and the candidate on
the same holdout.

In [ ]:
mae_old = mean_absolute_error(holdout["actual_min"],
                              model.predict(holdout[FEATURES]))

candidate_a = LinearRegression().fit(train[FEATURES],
                                     train["actual_min"])
mae_a = mean_absolute_error(holdout["actual_min"],
                            candidate_a.predict(holdout[FEATURES]))

print(f"same {len(holdout)} held-out orders:")
print(f"  old model      MAE {mae_old:.2f}")
print(f"  candidate A    MAE {mae_a:.2f}")

A big improvement: the error halved. In L28 this step alone was
enough to promote. Before we celebrate, the gate.

### Step 7 --- The gate, as code

Two conditions, both required.

**Beat the bar.** The product promised the launch quality. We
allow a quarter more: `BAR = 1.25 x launch MAE`. Note that this is
*stricter* than the alert threshold (1.5x). If you promoted at the
same level you alert at, one ordinary bad week would page you
again the day after the promotion.

**Beat the old model.** On the same holdout. Otherwise the retrain
is churn.

Written as a function, so that nobody can argue with it later.

In [ ]:
BAR = round(launch_mae * 1.25, 2)


def gate(mae_new, mae_old, bar):
    """Promote only if the new model beats the bar AND the old model."""
    if mae_new < bar and mae_new < mae_old:
        return "promote"
    return "hold"


def explain_gate(name, mae_new, mae_old, bar):
    """Print the two conditions and the verdict, in plain words."""
    print(f"{name} at the gate:")
    print(f"  beats the bar ({bar:.2f})?        "
          f"{'yes' if mae_new < bar else 'NO'}   ({mae_new:.2f})")
    print(f"  beats the old model ({mae_old:.2f})? "
          f"{'yes' if mae_new < mae_old else 'NO'}")
    verdict = gate(mae_new, mae_old, bar)
    print(f"  verdict: {verdict.upper()}")
    return verdict


verdict_a = explain_gate("candidate A", mae_a, mae_old, BAR)

**Hold.** Candidate A beats the old model comfortably and still
misses the bar. The gate did its job: "better than before" and
"good enough" are different questions, and only the second one
matters to a customer waiting for dinner.

### Step 8 --- Where candidate A still fails: the audit, again

P13's habit: split the error by group. If the same group is still
worst served, the retrain did not learn the rule --- it only
shifted the average.

In [ ]:
audit = holdout.copy()
audit["trip"] = pd.cut(audit["distance_km"], bins=[0, 5, 12, 100],
                       labels=["short (<5km)", "medium (5-12km)",
                               "long (>12km)"])
audit["old"] = (audit["actual_min"]
                - model.predict(audit[FEATURES])).abs()
audit["A"] = (audit["actual_min"]
              - candidate_a.predict(audit[FEATURES])).abs()

by_trip = (audit.groupby("trip", observed=True)
           .agg(orders=("old", "size"), old=("old", "mean"),
                A=("A", "mean"))
           .round(2))
print(by_trip.to_string())

Read the last row against the first. Candidate A made long trips
three times better --- and made short trips *worse*. A straight
line cannot be right at both ends of a road that bends at 12 km.
It settled for being wrong everywhere by a little.

### Step 9 --- Candidate B: teach the model the highway rule

The audit in P13 found the rule: kilometres beyond 12 cost extra.
Give the model a column that says exactly that. `highway_km` is
zero for a city trip and the distance past 12 km for a highway
trip.

The column is computed **inside** the model, by a small pipeline
step, so the service still sends the same four fields it always
has. Nothing outside the model needs to know.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer


def add_highway_km(X):
    """The rule the audit found: kilometres beyond 12 cost extra."""
    X = X.copy()
    X["highway_km"] = np.maximum(X["distance_km"] - 12.0, 0.0)
    return X


candidate_b = make_pipeline(FunctionTransformer(add_highway_km),
                            LinearRegression())
candidate_b.fit(train[FEATURES], train["actual_min"])
mae_b = mean_absolute_error(holdout["actual_min"],
                            candidate_b.predict(holdout[FEATURES]))

learned = candidate_b[-1]
for name, coef in zip(FEATURES + ["highway_km"], learned.coef_):
    print(f"  {name:<14} {coef:6.2f} minutes per unit")
print()
verdict_b = explain_gate("candidate B", mae_b, mae_old, BAR)

**Promote.** The model found the highway rule on its own: about
1.8 extra minutes for every kilometre past 12, which is exactly
the number hidden in the traffic generator. Both conditions hold,
so this candidate may go forward.

### Step 10 --- Draw what each model still gets wrong

For every held-out order, the **miss**: actual minutes minus
predicted. A healthy model's misses are a flat cloud around zero.
A slope or a bend in the cloud is a rule the model has not
learned. Three panels, one per model, same orders.

In [ ]:
candidates = [("old model", model, "#9E3B33"),
              ("candidate A", candidate_a, "#B4703A"),
              ("candidate B", candidate_b, "#2C5F79")]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for ax, (name, m, colour) in zip(axes, candidates):
    miss = holdout["actual_min"] - m.predict(holdout[FEATURES])
    ax.scatter(holdout["distance_km"], miss, s=14, color=colour,
               alpha=0.75)
    ax.axhline(0, color="#6B7785", linewidth=1)
    ax.axvline(12, color="#6B7785", linestyle=":", linewidth=1)
    ax.set_title(f"{name}: MAE {miss.abs().mean():.2f}", fontsize=10)
    ax.set_xlabel("distance (km)")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("actual minus predicted (min)")
fig.suptitle("A pattern in the misses is a rule the model has not "
             "learned", fontsize=11)
plt.tight_layout()
plt.show()

Left: the old model's misses bend sharply upward past the dotted
12 km line --- it does not know the highway exists. Middle:
candidate A tilts the whole cloud instead, so it is now low on the
left and high on the right. Right: candidate B is flat. That
flatness is what "learned the rule" looks like.

### Step 11 --- Audit all three models, group by group

P13's question, asked one last time before we promote anything:
**accurate for whom?** The same three trip-length groups, now with
candidate B added.

Watch two things: whether each group goes **under the bar**, and
how big the gap is between the best-served and worst-served group.
We call that gap the **fairness gap**: worst group's error divided
by best group's error. A gap of 1.0 means everyone is served
equally.

In [ ]:
audit["B"] = (audit["actual_min"]
              - candidate_b.predict(audit[FEATURES])).abs()
all_three = (audit.groupby("trip", observed=True)
             .agg(orders=("old", "size"), old=("old", "mean"),
                  A=("A", "mean"), B=("B", "mean")))
print(all_three.round(2).to_string())
print()
for name in ["old", "A", "B"]:
    gap = all_three[name].max() / all_three[name].min()
    print(f"fairness gap, {name:<3}: {gap:4.1f}x  "
          f"(worst group {all_three[name].idxmax()})")

x = np.arange(len(all_three))
fig, ax = plt.subplots(figsize=(9, 3.8))
for offset, (name, colour) in zip(
        (-0.27, 0, 0.27),
        [("old", "#9E3B33"), ("A", "#B4703A"), ("B", "#2C5F79")]):
    label = "old model" if name == "old" else f"candidate {name}"
    ax.bar(x + offset, all_three[name], width=0.26, color=colour,
           label=label)
ax.axhline(BAR, color="#6B7785", linestyle="--", linewidth=1,
           label=f"bar {BAR:.2f}")
ax.set_xticks(x, [f"{trip}\nn={n}" for trip, n in
                  zip(all_three.index, all_three["orders"])])
ax.set_ylabel("MAE on the holdout (minutes)")
ax.set_title("Same held-out orders, split by trip length",
             fontsize=10)
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

**Read each group left to right: red, orange, blue.**

- **Long trips:** the old model is far above the bar. A and B both
  bring it down; B brings it under.
- **Short and medium trips:** A is *taller* than the old model. A
  paid for the long-trip fix with the customers it was already
  serving well. B is as low as the old model, or lower.

Now look at the fairness gaps printed above the chart. **A has the
smallest gap of the three** --- and not one of its groups is under
the bar. A got its small gap by **levelling down**: it made the
best-served customers worse until they matched the rest.

B's gap is a little larger, but **every** group is under the bar,
and no group is worse off than under the old model. That is
**levelling up**.

So a fairness gap on its own can mislead you. Always read it next
to the errors themselves: "equal" is not the goal if everyone is
equally badly served.

### Step 12 --- Would the gate decide the same on a different exam?

The holdout is 120 orders. A different 120 might have given
different numbers. So before you promote, ask: **was the verdict
luck?**

The trick is called the **bootstrap**. Make a new exam paper by
picking 120 orders from the holdout *at random, with repeats
allowed*. Score every model on it. Do that 1,000 times. If a model
beats the bar on almost every re-sit, its pass was not luck.

The re-sits use their own random generator, so the traffic in
later steps is not disturbed.

In [ ]:
resit_rng = np.random.default_rng(0)
RESITS = 1000

y = holdout["actual_min"].to_numpy()
models = {"old model": model, "candidate A": candidate_a,
          "candidate B": candidate_b}
misses = {name: np.abs(y - m.predict(holdout[FEATURES]))
          for name, m in models.items()}

# each row of `papers` is one re-sit: 120 order positions
papers = resit_rng.integers(0, len(y), size=(RESITS, len(y)))
resit_mae = {name: miss[papers].mean(axis=1)
             for name, miss in misses.items()}

print(f"{'':<13}{'MAE':>6}{'middle 90% of re-sits':>24}"
      f"{'re-sits under the bar':>24}")
for name, scores in resit_mae.items():
    low, high = np.percentile(scores, [5, 95])
    passed = int((scores < BAR).sum())
    print(f"{name:<13}{misses[name].mean():>6.2f}"
          f"{f'{low:.2f} to {high:.2f}':>24}"
          f"{f'{passed} of {RESITS}':>24}")
b_beats_old = int((resit_mae["candidate B"]
                   < resit_mae["old model"]).sum())
print()
print(f"candidate B beat the old model on {b_beats_old} of "
      f"{RESITS} re-sits")

fig, ax = plt.subplots(figsize=(9, 3.4))
for name, colour in [("old model", "#9E3B33"),
                     ("candidate A", "#B4703A"),
                     ("candidate B", "#2C5F79")]:
    ax.hist(resit_mae[name], bins=40, color=colour, alpha=0.75,
            label=name)
ax.axvline(BAR, color="#6B7785", linestyle="--",
           label=f"bar {BAR:.2f}")
ax.set_xlabel("MAE on one re-sit (minutes)")
ax.set_ylabel("number of re-sits")
ax.set_title("1,000 re-sits of the same exam", fontsize=10)
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

Three humps, and the bar between two of them.

- **Candidate B's** hump sits left of the bar; only its very
  right-hand edge touches it. It passed on almost every re-sit:
  the promote verdict is not luck.
- **Candidate A's** hump sits right of the bar; only its very
  left-hand edge touches it. It failed on almost every re-sit: the
  hold verdict is not bad luck either.
- The **old model** is far to the right, as P13 told us.

When would you worry? If a hump **straddled** the bar --- passing
on, say, 60% of re-sits. Then the honest verdict is "not sure yet":
collect another week of orders and judge again.

### Step 13 --- Put the launch model into the registry as version 1

Now MLflow, exactly as in P06: a database file, an experiment, a
registered model name. First the launch model goes in as
**version 1** with the alias `@champion`, because that is what
has been serving customers.

`pip_requirements` tells MLflow which libraries the model needs.
Without it MLflow spends half a minute working that out on its
own, every time.

In [ ]:
import shutil

import mlflow
import mlflow.sklearn
from mlflow import MlflowClient

db_path = (WORK / "mlflow.db").resolve()
db_path.unlink(missing_ok=True)          # fresh store for teaching
shutil.rmtree("mlartifacts", ignore_errors=True)

mlflow.set_tracking_uri(f"sqlite:///{db_path.as_posix()}")
mlflow.set_experiment("delivery-time")
MODEL_NAME = "delivery-time-model"
NEEDS = ["scikit-learn", "pandas", "numpy"]
# three example rows, as floats so MLflow does not warn about
# integer columns that might one day be missing
EXAMPLE = X_train.head(3).astype(float)

with mlflow.start_run(run_name="launch-champion"):
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("window", "training-600")
    mlflow.log_metric("mae_minutes", round(launch_mae, 2))
    mlflow.sklearn.log_model(model, name="model",
                             input_example=EXAMPLE,
                             pip_requirements=NEEDS,
                             registered_model_name=MODEL_NAME)

client = MlflowClient()
client.set_registered_model_alias(MODEL_NAME, "champion", 1)
print("version 1 registered; @champion -> 1")

### Step 14 --- Register version 2 and move the alias, keeping version 1

Candidate B passed the gate. Log the run with **both** numbers
from the judging, register the model as **version 2**, and move
`@champion` to it. Version 1 stays in the registry, untouched.

One new argument. MLflow 3 saves scikit-learn models in a safe
format that refuses to store a function it does not know --- and
our pipeline contains `add_highway_km`. `serialization_format=
"cloudpickle"` packs the function in with the model. MLflow prints
a warning about this; read it, it is fair.

In [ ]:
with mlflow.start_run(run_name="retrain-weeks-9-12"):
    mlflow.log_param("model_type", "LinearRegression+highway_km")
    mlflow.log_param("window", "weeks-9-12")
    mlflow.log_metric("mae_minutes", round(mae_b, 2))
    mlflow.log_metric("mae_old_same_holdout", round(mae_old, 2))
    mlflow.log_metric("bar", BAR)
    mlflow.sklearn.log_model(candidate_b, name="model",
                             input_example=EXAMPLE,
                             pip_requirements=NEEDS,
                             serialization_format="cloudpickle",
                             registered_model_name=MODEL_NAME)

client.set_registered_model_alias(MODEL_NAME, "champion", 2)

print("registry now holds:")
for mv in sorted(client.search_model_versions(f"name='{MODEL_NAME}'"),
                 key=lambda mv: int(mv.version)):
    tag = "  <- @champion" if "champion" in mv.aliases else ""
    print(f"  version {mv.version}  run {mv.run_id[:8]}...{tag}")

Two versions, one alias. Rollback is the alias moved back to 1:
one line, minutes, no rebuild. You will do exactly that in Task
T2.

### Step 15 --- Serving asks for @champion, and gets the new model

The service from P07 loads its model by name. Change the name to
the alias and the service picks up every promotion without a code
change. Here is what it would answer for an 18 km order, before
and after.

In [ ]:
champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")

long_order = pd.DataFrame([{
    "distance_km": 18.0,
    "prep_time_min": 20,
    "traffic_level": 2,
    "rain": 0,
}])

print("an 18 km order, 20 min prep, medium traffic, dry:")
print(f"  version 1 (launch)  -> {float(model.predict(long_order)[0]):.1f} min")
print(f"  @champion now       -> {float(champion.predict(long_order)[0]):.1f} min")
print()
print("loaded:", type(champion).__name__,
      "-- the pipeline, highway rule included")

Eleven more minutes for the same order. The old model was
promising a time the rider could not keep; the customer on the
edge of town is the one who felt it.

### Step 16 --- Old and new, across every distance

One order is an anecdote. Ask both versions about **every** distance
from 0.5 to 22 km, with everything else held fixed (20 min prep,
medium traffic, dry), and draw the answers as two lines.

The thin dark dashed line is the **true rule** --- the formula hidden in
Step 2's traffic generator, without the random noise. In real life
you never get to see this line. Here you do, so you can check who is
right.

In [ ]:
km = np.linspace(0.5, 22, 88)
grid = pd.DataFrame({"distance_km": km, "prep_time_min": 20,
                     "traffic_level": 2, "rain": 0})
v1_line = model.predict(grid)
v2_line = champion.predict(grid)
truth = (6.0 + 3.1 * km + 0.65 * 20 + 4.2 * 2
         + 1.8 * np.maximum(km - 12.0, 0.0))

print(f"{'km':>5}{'version 1':>11}{'version 2':>11}{'true rule':>11}")
for d in (3, 6, 9, 12, 15, 18, 21):
    i = int(np.argmin(np.abs(km - d)))
    print(f"{d:>5}{v1_line[i]:>11.1f}{v2_line[i]:>11.1f}"
          f"{truth[i]:>11.1f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(km, v1_line, color="#9E3B33", linewidth=2,
        label="version 1 (launch)")
ax.plot(km, v2_line, color="#2C5F79", linewidth=4, alpha=0.6,
        label="version 2 (@champion)")
ax.plot(km, truth, color="#333333", linestyle="--", linewidth=1.2,
        label="true rule (hidden in the generator)", zorder=3)
ax.axvline(12, color="#6B7785", linestyle=":", linewidth=1)
ax.fill_between(km, v1_line, v2_line, where=km > 12,
                color="#9E3B33", alpha=0.10)
ax.annotate("minutes the old model\nwas promising wrongly",
            xy=(19.5, 95), xytext=(17, 62), fontsize=8,
            color="#9E3B33", ha="center",
            arrowprops=dict(arrowstyle="->", color="#9E3B33"))
ax.set_xlabel("distance_km")
ax.set_ylabel("predicted minutes")
ax.set_title("Same order, every distance", fontsize=10)
ax.legend(fontsize=8, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

**Left of the dotted line**, all three lines lie on top of each
other. Inside the city the old model was never wrong, and the new
one has not broken anything there.

**Right of it**, the old model keeps going straight while the true
rule bends upward. The shaded wedge between red and blue is the
time the old model was under-promising, and it grows with every
kilometre. Version 2 bends with the truth.

This is why a retrain needs the right rows *and*, sometimes, the
right feature: no amount of data would let a straight line bend.

### Step 17 --- Close the loop: four more weeks with the new champion

The retrain is only a success if the error comes back down *on
traffic the model has never seen*. Weeks 13 to 16 arrive. Score
both versions on them and draw the whole story, week 1 to 16.

In [ ]:
future = pd.concat([week_of_orders(w) for w in range(13, 17)],
                   ignore_index=True)
future["v1_error"] = (future["actual_min"]
                      - model.predict(future[FEATURES])).abs()
future["v2_error"] = (future["actual_min"]
                      - champion.predict(future[FEATURES])).abs()
after = future.groupby("week")[["v1_error", "v2_error"]].mean().round(2)
print(after.to_string())

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(weekly.index, weekly["mae"], marker="o", linewidth=2,
        color="#9E3B33", label="version 1, serving")
ax.plot(after.index, after["v1_error"], marker="o", linestyle="--",
        color="#9E3B33", alpha=0.55, label="version 1, had we done nothing")
ax.plot(after.index, after["v2_error"], marker="o", linewidth=2,
        color="#2C5F79", label="version 2, serving")
ax.axvline(12.5, color="#6B7785", linestyle=":")
ax.text(12.42, 4.3, "retrain + promote", rotation=90,
        ha="right", va="center", fontsize=8, color="#6B7785")
ax.axhline(BAR, linestyle="--", color="#6B7785", linewidth=1,
           label=f"bar {BAR:.2f}")
ax.set_xticks(range(1, 17))
ax.set_xlabel("week")
ax.set_ylabel("MAE (minutes)")
ax.set_title("Monitoring found it, the retrain fixed it, "
             "monitoring confirms it")
ax.legend(fontsize=8, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

The blue line sits under the bar for all four weeks. The faded red
line is the counterfactual: what customers would still be getting
if nobody had looked. This chart is the whole course on one
screen, and monitoring does not stop here --- week 17 might drift
again.

### Step 18 --- Write it down: the promotion record

P13 ended with an incident note: what broke. P14 ends with its
partner, the **promotion record**: what we changed, on what
evidence, and how to undo it.

Every number below is read from a variable computed earlier, so
the record cannot disagree with the charts. The last line is left
blank on purpose. A pipeline can fill in every number; only a
person can sign.

In [ ]:
serving = client.get_model_version_by_alias(MODEL_NAME, "champion")
lines = [
    f"PROMOTION RECORD -- {MODEL_NAME}",
    "",
    f"CHANGE    @champion moved from version 1 to version "
    f"{serving.version}. Version 1 kept.",
    f"WHY       Error alarm fired in week {first_alert}; error "
    f"settled near {weekly.loc[9:12, 'mae'].mean():.1f} min "
    f"(launch {launch_mae:.2f}).",
    f"DATA      Weeks 9-12 ({len(train)} train, {len(holdout)} "
    f"sealed holdout); {beyond_window:.0%} of trips over 12 km.",
    f"FIX       Added highway_km; learned "
    f"{learned.coef_[-1]:.2f} min per km beyond 12.",
    f"GATE      New {mae_b:.2f} vs bar {BAR:.2f} and old "
    f"{mae_old:.2f} -> PROMOTE "
    f"(under the bar on "
    f"{int((resit_mae['candidate B'] < BAR).sum())} of "
    f"{RESITS} re-sits).",
    f"FAIRNESS  Long trips {all_three.loc['long (>12km)', 'old']:.2f}"
    f" -> {all_three.loc['long (>12km)', 'B']:.2f} min; short "
    f"trips {all_three.loc['short (<5km)', 'old']:.2f} -> "
    f"{all_three.loc['short (<5km)', 'B']:.2f}.",
    f"AFTER     Weeks 13-16 MAE {after['v2_error'].min():.2f} to "
    f"{after['v2_error'].max():.2f}; "
    f"{(after['v2_error'] < BAR).sum()} of 4 weeks under the bar.",
    f"ROLLBACK  client.set_registered_model_alias("
    f"'{MODEL_NAME}', 'champion', 1)",
    "APPROVED  ______________________  (a name and a date, "
    "not a team)",
]
record = "\n".join(lines)

print(record)
(WORK / "promotion_record.txt").write_text(record, encoding="utf-8")
print()
print("saved to", WORK / "promotion_record.txt")

Read it as the person who has to approve it. It says what changed,
why, on which rows, with what evidence, whether anyone was made
worse off, and how to undo it in one line. That is everything a
reviewer needs, and nothing they do not.

Note the AFTER line counts the weeks under the bar. That claim
was *checked* in Step 17, not assumed. If week 17 drifts again, the
same monitoring from P13 will say so, and this record tells the
next person exactly where to start.

---

# Your turn

The walkthrough above is finished. Now you write some code.

There are **3 tasks**. Each one is small. Each one has a hint.
Do them in order.

Where you see `# TODO`, replace that line with your own code. Do not delete
the variable name on the left of the `=` sign --- the self-check at the
bottom looks for exactly that name.

When you have tried all three, run the **self-check** cell at the end. It
prints a table telling you which tasks are correct. You can run it as many
times as you like.

### Task T1 --- The stale window


A teammate argues the retrain should use weeks **1 to 4** --- "that
is when the model was healthy". Test the idea honestly.

1. Train a `LinearRegression` on the live orders from weeks 1-4 only,
   into `T1_model`.
2. Judge it on the **same** `holdout` as everything else, into
   `T1_mae`.
3. Put the gate's verdict against the old model in `T1_verdict`.


> **Hint.** `stale = live[live["week"] <= 4]`, then fit on `stale[FEATURES]` and `stale["actual_min"]`. Judge with `mean_absolute_error(holdout["actual_min"], T1_model.predict(holdout[FEATURES]))`, and the verdict is `gate(T1_mae, mae_old, BAR)`.

In [ ]:
# TODO: a LinearRegression trained on weeks 1-4 of `live` only
T1_model = None

# TODO: its MAE on the SAME holdout as Step 6
T1_mae = None

# TODO: the gate's verdict against the old model
T1_verdict = None

print("weeks 1-4 model on the holdout:", T1_mae, "->", T1_verdict)

### Task T2 --- Roll back, then forward again


Monday morning: a bug report blames version 2. Prove you can undo the
promotion in one line, and redo it in one more.

1. Move the alias `champion` back to version **1**.
2. Load whatever `@champion` points at now into `T2_model`, and put
   its version (as a string) in `T2_version` and its prediction for
   `long_order`, rounded to one decimal, in `T2_prediction`.
3. Move the alias forward to version **2** again.


> **Hint.** `client.set_registered_model_alias(MODEL_NAME, "champion", 1)` moves it. `client.get_model_version_by_alias(MODEL_NAME, "champion").version` tells you where it points. Load with the same `models:/...@champion` string as Step 15.

In [ ]:
# TODO: move @champion back to version 1

# TODO: load @champion, record its version and its 18 km prediction
T2_model = None
T2_version = None
T2_prediction = None

# TODO: move @champion forward to version 2 again

print("rolled back to version", T2_version, "->", T2_prediction, "min")
print("now serving version",
      client.get_model_version_by_alias(MODEL_NAME, "champion").version)

### Task T3 --- A challenger of your own, through the whole gate


Would a random forest (P03) do better than a straight line, now that
it too gets the highway column? Find out the honest way.

1. Build `T3_model`: a pipeline of `FunctionTransformer(add_highway_km)`
   and `RandomForestRegressor(n_estimators=200, max_depth=8,
   random_state=42)`, trained on `train`.
2. Judge it on the **same** `holdout`, into `T3_mae`.
3. Put the gate's verdict against the **current champion**
   (`mae_b`) in `T3_verdict`.
4. Log an MLflow run named `challenger-forest` with the metric
   `mae_minutes`, whatever the verdict.
5. Only if the verdict is `"promote"`: log and register the model in
   that run (remember `serialization_format="cloudpickle"`) and move
   the alias to version 3. If it is `"hold"`, leave the alias alone.

`hold` is a perfectly good answer. Say what you found.


> **Hint.** Copy Step 9 for the pipeline, swapping `LinearRegression()` for the forest. The run needs `with mlflow.start_run(run_name="challenger-forest"):` and `mlflow.log_metric("mae_minutes", T3_mae)`. Only register inside the `if T3_verdict == "promote":` branch.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# TODO: the forest, with the highway column, trained on `train`
T3_model = None

# TODO: its MAE on the SAME holdout
T3_mae = None

# TODO: the gate's verdict against the current champion (mae_b)
T3_verdict = None

# TODO: log a run named "challenger-forest" with metric mae_minutes;
#       register and promote to version 3 ONLY if the verdict is "promote"

print("challenger:", T3_mae, "->", T3_verdict)

---

## Self-check

Run the cell below to mark your work.

In [ ]:
# ------------------------------------------------------------------
# SELF-CHECK --- run this when you have attempted the tasks above.
# It never breaks your notebook. A task you have not done yet simply
# shows FAIL.
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    """Evaluate one graded condition without ever raising."""
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check('T1 | T1_model is a trained LinearRegression', lambda: hasattr(T1_model, 'coef_') and len(T1_model.coef_) == 4)
_check('T1 | T1_mae is its error on the shared holdout', lambda: abs(float(T1_mae) - mean_absolute_error(holdout['actual_min'], T1_model.predict(holdout[FEATURES]))) < 1e-6)
_check('T1 | the stale window does worse than the recent one', lambda: float(T1_mae) > float(mae_a))
_check('T1 | T1_verdict is what the gate says', lambda: T1_verdict == gate(float(T1_mae), mae_old, BAR))
_check('T2 | the alias really pointed at version 1 during the rollback', lambda: str(T2_version) == '1')
_check("T2 | the rolled-back prediction is the launch model's", lambda: abs(float(T2_prediction) - float(model.predict(long_order)[0])) < 0.06)
_check('T2 | the alias was moved forward again afterwards', lambda: str(T2_version) == '1' and str(client.get_model_version_by_alias(MODEL_NAME, 'champion').version) != '1')
_check("T3 | T3_mae is the challenger's error on the shared holdout", lambda: abs(float(T3_mae) - mean_absolute_error(holdout['actual_min'], T3_model.predict(holdout[FEATURES]))) < 1e-6)
_check('T3 | T3_verdict is what the gate says against the champion', lambda: T3_verdict == gate(float(T3_mae), mae_b, BAR))
_check('T3 | the attempt was logged as challenger-forest', lambda: abs(float(mlflow.search_runs(experiment_names=['delivery-time'], filter_string="tags.mlflow.runName = 'challenger-forest'")['metrics.mae_minutes'].iloc[0]) - float(T3_mae)) < 0.01)
_check('T3 | the alias moved only if the gate said promote', lambda: str(client.get_model_version_by_alias(MODEL_NAME, 'champion').version) == ('3' if T3_verdict == 'promote' else '2'))

print("==================================================================")
print("SELF-CHECK   Practical 14 --- Teaching the Model the New Suburb")
print("==================================================================")
for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")
print("------------------------------------------------------------------")
_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("==================================================================")
if _passed == len(_results):
    print("Well done. Save the notebook and submit it.")
else:
    print("Read the FAIL lines above, fix those tasks, run this cell again.")

---

## What to submit

1. This notebook, with every cell run and every chart visible.
2. In a markdown cell, the two numbers the gate compared for candidate B, and one sentence on why candidate A was held even though it beat the old model.
3. In a markdown cell, the exact one-line command you would run to roll back, and who in a real team should be allowed to run it. Step 18's promotion record shows the shape; write it in your own words.

Name your file `PXX_<your-roll-number>.ipynb` before you upload it.

### How this practical is marked

| What is marked | Marks |
|---|---|
| Walkthrough run end to end, all charts produced | 3 |
| Task T1 --- the stale window judged on the same holdout | 2 |
| Task T2 --- rollback and re-promotion through the alias | 2 |
| Task T3 --- a challenger taken through the full gate | 3 |
| **Total** | **10** |

---

## Read more

- MLflow --- Model Registry: aliases --- <https://mlflow.org/docs/latest/model-registry.html>
- MLflow --- Saving scikit-learn models (serialization formats) --- <https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html>
- scikit-learn --- Pipelines and FunctionTransformer --- <https://scikit-learn.org/stable/modules/compose.html>
- Evidently --- When to retrain --- <https://www.evidentlyai.com/ml-in-production/model-retraining>
- Google --- Rules of ML (rule 37: measure training/serving skew) --- <https://developers.google.com/machine-learning/guides/rules-of-ml>